In [2]:
import gzip
import pickle
import pandas as pd
import numpy as np
import tabix
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Select a model
models_list = ['DNABERT', 'nucleotide-transformer']
model = 'DNABERT'

# Select a submodel
submodels_list = ['finetuned', 'pretrained', 'random_init', 'random_pretrained'] #NOTE - random_pretrained is only available for DNABERT
submodel = 'finetuned'

# Select a dataset
datasets_list = ['TATA', 'enhancers']
dataset = 'TATA'

# Number of layers
nlayers = 12 if model == 'DNABERT' else 29
nheads = 12 if model == 'DNABERT' else 16

# Pathways are highlighted with the hastag #REVIEW

In [ ]:
if model == "DNABERT":
    # Function for DNABERT models
    def transform_values(values):
        new_values:list = np.array([np.mean(values[i:i+6]) for i in range(len(values)-5)])
        return(new_values)
else:
    # Function for nucleotide-transformer models
    def transform_values(values):
        n = len(values)
        r = n % 6 
        new_values = np.array([np.mean(values[i:i+6]) for i in range(0, n - r, 6)])
        if r > 0:
            new_values = np.append(new_values, values[-r:])
        return(new_values)

# Extract layer information

In [ ]:
def kmer2seq(kmers_list:list) -> str:
    first:bool = True
    sequence:str = ''
    for token in kmers_list:
        if first:
            sequence = token
            first = False
        elif (token != '[SEP]' and token != '[PAD]'):
            sequence += token[-1]
        else:
            break
    return(sequence)

# Open the original test datatset
blat_query_seq_results_filtered = pd.read_csv(f'test_datasets/{dataset}_test.csv') #REVIEW - Check that the path is correct
blat_sequences:list = list(blat_query_seq_results_filtered['Sequence'])

# Extract layer attention scores
results_dict:dict = {}
for layer in range(nlayers):
    with open(f'{model}/attention_scores/{submodel}/layer{layer}.p', 'rb') as f: #REVIEW - Check that the path is correct
        results:dict = pickle.load(f)
    for head in range(nheads):
        for example in range(len(results[head])):
            if model == "DNABERT":
                sequence:str = kmer2seq(results[head][example][1])
            else:
                sequence:str = ''.join([kmer for kmer in results[head][example][1] if (kmer != '<cls>' and kmer != '<pad>')])
            if sequence not in blat_sequences:
                continue
            if layer == 0 and head ==0:
                results_dict[sequence] = {}
                results_dict[sequence]['kmers'] = [kmer for kmer in results[head][example][1] if (kmer not in ['[SEP]', '[PAD]', '<cls>', '<pad>'])]
            kmers_vector_length = len(results_dict[sequence]['kmers'])
            results_dict[sequence][f'layer{layer}-head{head}'] = np.array(results[head][example][0][1:kmers_vector_length+1])

# Get sequences information
sequences:list = list(results_dict.keys())
indexes:list = [blat_query_seq_results_filtered[blat_query_seq_results_filtered['Sequence'] == sequence].index[0] for sequence in sequences]

# Analysis

In [ ]:
# Extract conservation information (PhyloP)
phyloP:tabix = tabix.open('databases/hg38.phyloP100way.sorted.combined.bed.gz') #REVIEW - Check that the path is correct
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from phyloP database with tabix
    records:object = phyloP.querys(f"chr{chrom}:{start}-{end}")
    values:list = []
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values.append(float(record[3]))
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    ## Add this layer to the original one
    if len(values) != kmers_vector_length+5: #Very repetitive regions may not have phyloP or an incomplete track
        results_dict[sequence]['phyloP'] = np.array(np.full(kmers_vector_length, np.nan))
    else:
        results_dict[sequence]['phyloP'] = transform_values(values)

del(phyloP, idx, sequence, chrom, start, end, records, values, record, position, kmers_vector_length)


In [ ]:
# Annotate TSS regions
tss_db:tabix = tabix.open("databases/refTSS_v4.1_human_coordinate.hg38.bed.txt.gz") #REVIEW - Check that the path is correct
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from refTSS database with tabix
    records:object = tss_db.querys(f"chr{chrom}:{start}-{end}")
    tss_region:list = []
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    for record in records:
        tss_region.extend([i for i in range(int(record[1]), int(record[2])+1)]) #Can be more than one TSS described for each region
    if len(tss_region) == 0: #If there isn't a TSS in the region, fill the layer with NAs
        results_dict[sequence]['TSS'] = np.array(np.full(kmers_vector_length, np.nan))
    else:
        query_region = [i for i in range(start, end+1)]
        tss_vector = []
        for pos in query_region:
            tss_vector.extend([1 if pos in tss_region else 0])
        ## Add this layer to the original one
        results_dict[sequence]['TSS'] = transform_values(tss_vector[1:])

del(tss_db, idx, sequence, chrom, start, end, records, tss_region, record, pos, query_region, tss_vector)

In [7]:
# Annotate GC content
for sequence in sequences:
    kmer_list:list = results_dict[sequence]['kmers']
    gc_vector:list = []
    for kmer in kmer_list:
        ## Change C and G to 1, and A and T to 0
        gc_sequence = kmer.replace('C', '1').replace('G', '1').replace('A', '0').replace('T', '0')
        gc_sequence = [int(i) for i in gc_sequence]
        gc_vector.append(np.mean(gc_sequence))
    
    ## Add this layer to the original one
    results_dict[sequence]['GC'] = np.array(gc_vector)

del(sequence, kmer_list, gc_vector, kmer, gc_sequence)

In [ ]:
# Extract TF information (TFlink)
tflink:tabix = tabix.open('databases/TFlink_hg38.sorted.bed.gz') #REVIEW - Check that the path is correct
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from TFlink database with tabix
    records:object = tflink.querys(f"{chrom}:{start}-{end}")
    tf_positions:dict = {} #Each entry will be the genomic regions for each transcript factor
    for record in records:
        tf:str = record[3]
        if tf in tf_positions.keys(): #Each TF can be detected in different regions, but we want an entry per TF
            tf_positions[tf].extend([i for i in range(int(record[1]), int(record[2])+1)])
        else:
            tf_positions[tf] = [i for i in range(int(record[1]), int(record[2])+1)]
    ## Add each TF as a new key in the sequences dictionary
    region:list = [i for i in range(start, end+1)]
    for tf in tf_positions:
        region_scores:list = []
        for pos in region:
            region_scores.extend([1 if pos in tf_positions[tf] else 0])
        results_dict[sequence][tf] = transform_values(region_scores[1:])

del(tflink, idx, sequence, chrom, start, end, records, tf, record, tf_positions, region, region_scores, pos)

# Since not all the TF are detected in all the sequences, we need to fill those with NAs
TF_families:pd.DataFrame = pd.read_csv('databases/TF_families.csv', sep=",") #REVIEW - Check that the path is correct
for sequence in results_dict:
    for tf in TF_families['TF']:
        if tf not in results_dict[sequence].keys():
            kmers_vector_length:int = len(results_dict[sequence]['kmers'])
            results_dict[sequence][tf] = np.array(np.full(kmers_vector_length, np.nan))

del(sequence, tf)

# Combine TF matrix by families
uniqueTF_families:list = list(TF_families['Family'].unique())
for sequence in results_dict:
    for family in uniqueTF_families:
        idx:pd.Index = TF_families[TF_families['Family']==family].index
        results_dict[sequence][family] = np.nansum(np.vstack([results_dict[sequence][TF_families['TF'][i]] for i in idx]), axis=0)
        
del(sequence, family, idx, uniqueTF_families)

# Remove single TFs from the results dictionary
for sequence in results_dict:
    for tf in TF_families['TF']:
        try:
            results_dict[sequence].pop(tf)
        except KeyError:
            continue

del(sequence, tf, TF_families)

In [ ]:
# Check if the sequence belong to a repeat element
repeat_db:tabix = tabix.open("databases/repeat_masker_hg38.bed.gz") #REVIEW - Check that the path is correct
repeat_families:pd.DataFrame = pd.read_csv('databases/repeat_families.csv', sep=",") #REVIEW - Check that the path is correct
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Create a list for each repeat element
    repeat_positions:dict = {} #Each entry will be the genomic regions for repeat element
    for repeat in repeat_families['Family']:
        repeat_positions[repeat] = []
    ## Retrieve information from repeat masker database with tabix
    records:object = repeat_db.querys(f"{chrom}:{start}-{end}")
    for record in records:
        try: #There are some repeats like LTR? or DNA? that we are not considering
            repeat_positions[record[4]].extend([i for i in range(int(record[1]), int(record[2])+1)])
        except KeyError:
            continue
    ## Add each repeat element as a new key in the sequences dictionary
    region:list = [i for i in range(start, end+1)]
    for repeat in repeat_positions:
        region_scores:list = []
        for pos in region:
            region_scores.extend([1 if pos in repeat_positions[repeat] else 0])
        results_dict[sequence][repeat] = transform_values(region_scores[1:])

del(repeat_db, idx, sequence, chrom, start, end, records, repeat, record, repeat_positions, region, region_scores, pos)

In [11]:
# Combine repeats by families
unique_repeat_families:list = list(repeat_families['Superfamily'].unique())
for sequence in results_dict:
    for family in unique_repeat_families:
        idx:pd.Index = repeat_families[repeat_families['Superfamily']==family].index
        results_dict[sequence][family] = np.nansum(np.vstack([results_dict[sequence][repeat_families['Family'][i]] for i in idx]), axis=0)
        
del(sequence, family, idx, unique_repeat_families)

In [12]:
# Remove single repeats from the results dictionary
for sequence in results_dict:
    for repeat in repeat_families['Family']:
        try:
            results_dict[sequence].pop(repeat)
        except KeyError:
            continue

del(sequence, repeat, repeat_families)

In [13]:
# Add positional features
for sequence in results_dict:
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    results_dict[sequence]['position'] = np.array([i for i in range(kmers_vector_length)])

del(sequence)

In [14]:
# Add sequence label
for sequence in results_dict.keys():
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    results_dict[sequence]['label'] = np.full(kmers_vector_length, blat_query_seq_results_filtered.loc[blat_query_seq_results_filtered["Sequence"] == sequence, "Label"].iloc[0])

del(sequence)

In [15]:
# Extract feature list to check
features_list:list = list(results_dict[list(results_dict.keys())[0]].keys())
features_list = [i for i in features_list if (i not in ['sequence', 'kmers', 'phyloP', 'TSS', 'GC', 'position', 'label']) and (not i.startswith('layer'))]

# Initialize the feature dict
features_dict:dict = {}
for feature in results_dict[list(results_dict.keys())[0]]:
    if feature not in []:
        features_dict[feature] = []

# Filter features present in < 5% of sequences
remove_features:list = []
for key in results_dict:
    for feature in features_list:
        features_dict[feature].append(0 if max(results_dict[key][feature]) == 0 else 1)
for feature in features_list:
    if (sum(features_dict[feature]) / len(features_dict[feature])) < 0.05:
        remove_features.append(feature)

# Remove lowly expressed features
for key in results_dict:
    for feature2remove in remove_features:
        del results_dict[key][feature2remove]

del(features_list, features_dict, feature, remove_features, key, feature2remove)

In [ ]:
with open(f'{model}/{model}_{submodel}_{dataset}_scores_and_bio_annotations.pkl', 'wb') as output: #REVIEW - Check that the path is correct
    pickle.dump(results_dict, output, protocol=pickle.HIGHEST_PROTOCOL)

del(output)

# Prepare results for correlation analysis

In [17]:
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
## Minmax normalization
scaler = MinMaxScaler()
results_dict_df:dict = {}
results_dict_df_norm = pd.DataFrame()
for gene, gene_dict in results_dict.items():
    gene_dict_df = pd.DataFrame(gene_dict)
    gene_dict_df = gene_dict_df[[col for col in gene_dict_df.columns if col.startswith('layer')]]
    gene_dict_df.columns = [f'{gene};{col}' for col in gene_dict_df.columns]
    results_dict_df[gene] = gene_dict_df
results_dict_df = pd.concat(results_dict_df.values(), axis=1)
for layer in range(nlayers):
    tmp_by_layer = results_dict_df[[col for col in results_dict_df.columns if f'layer{layer}' in col]]
    tmp_by_layer_values = tmp_by_layer.values.flatten().reshape(-1, 1)
    tmp_by_layer_values = np.nan_to_num(tmp_by_layer_values, nan = tmp_by_layer_values.mean())
    tmp_by_layer_values = scaler.fit_transform(tmp_by_layer_values)
    tmp_by_layer_norm = pd.DataFrame(tmp_by_layer_values.reshape(tmp_by_layer.shape), columns=tmp_by_layer.columns)
    results_dict_df_norm = pd.concat([results_dict_df_norm, tmp_by_layer_norm])
## Return NaN values
results_dict_df_norm = results_dict_df_norm.mask(results_dict_df.isna())
## Transforming back into a dictionary with the original shape
for col in results_dict_df_norm.columns:
    gene, layer_head = col.split(';')
    results_dict[gene][f'{layer_head}_minmax'] = np.array(results_dict_df_norm[col].dropna())

In [ ]:
# Convert dictionary into list of dictionaries
df:list = []
for key, subdict in results_dict.items():
    dict_by_row:dict = {'sequence': key}
    for subkey, values_list in subdict.items():
        dict_by_row[subkey] = ','.join(map(str, values_list))
    df.append(dict_by_row)

# Create DataFrame
df:pd.DataFrame = pd.DataFrame(df)

# Save the dataframe as a csv file with semicolon separator
with gzip.open(f'{model}/{model}_{submodel}_{dataset}_scores_and_bio_annotations_corformat.csv.gz', 'wt', newline='', encoding='utf-8') as f: #REVIEW - Check that the path is correct
    df.to_csv(f, sep=';', index=False)

del(key, subdict, subkey, dict_by_row, values_list, df, f)